# 01 — Exploratory Data Analysis
## WNYC Masculinity Survey

This notebook explores the raw survey data before any modeling. It covers:
- Dataset shape and column inventory
- Missing value audit
- Distributions of key demographic and attitudinal variables
- Geographic distribution of respondents
- Lifestyle frequency patterns (Q7)
- Worry profiles (Q8)
- Relationship between key variables

**Data:** `data/raw-responses.csv` — 1,615 respondents, 98 columns

**Q7 column mapping (per survey instrument):**
- q0007_0001: Ask a friend for professional advice
- q0007_0002: Ask a friend for personal advice
- q0007_0003: Express physical affection to male friends
- q0007_0004: Cry
- q0007_0005: Get in a physical fight
- q0007_0006: Have sexual relations with women
- q0007_0007: Have sexual relations with men
- q0007_0008: Watch sports
- q0007_0009: Work out
- q0007_0010: See a therapist
- q0007_0011: Feel lonely or isolated

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

df = pd.read_csv('../data/raw-responses.csv')
df.rename({'Unnamed: 0': 'ID'}, axis=1, inplace=True)

print(f"Shape: {df.shape}")
df.head(3)

## 1. Column Inventory & Data Types

In [ ]:
with pd.option_context('display.max_rows', None):
    print(df.dtypes)

## 2. Missing Value Audit

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_n': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_n'] > 0].sort_values('missing_n', ascending=False)

print(f"Columns with missing values: {len(missing_df)} of {df.shape[1]}")

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(missing_df.index[:25][::-1], missing_df['missing_pct'][:25][::-1], color='#EF5350')
ax.set_xlabel('% missing', fontsize=11)
ax.set_title('Top 25 Columns by Missing Value Rate', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 3. Key Variable Distributions

In [ ]:
q1_order = ['Very masculine', 'Somewhat masculine', 'Not very masculine', 'Not at all masculine']
q1_counts = df['q0001'].value_counts().reindex(q1_order).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(q1_counts.index, q1_counts.values, color='#1976D2')
axes[0].set_title('Q1: How masculine do you feel?', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(q1_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

q2_order = ['Very important', 'Somewhat important', 'Not too important', 'Not at all important']
q2_counts = df['q0002'].value_counts().reindex(q2_order).dropna()
axes[1].bar(q2_counts.index, q2_counts.values, color='#7B1FA2')
axes[1].set_title('Q2: How important is it others see you as masculine?', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(q2_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

q5_counts = df['q0005'].value_counts()
axes[0].bar(q5_counts.index, q5_counts.values, color=['#4CAF50', '#F44336', '#9E9E9E'][:len(q5_counts)])
axes[0].set_title('Q5: Society puts unhealthy pressure on men?', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(q5_counts.items()):
    axes[0].text(i, val + 8, f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

q17_counts = df['q0017'].value_counts()
axes[1].bar(q17_counts.index, q17_counts.values, color=['#2196F3', '#FF5722', '#9E9E9E'][:len(q17_counts)])
axes[1].set_title('Q17: Expected to make the first move?', fontweight='bold')
axes[1].set_ylabel('Count')
for i, (idx, val) in enumerate(q17_counts.items()):
    axes[1].text(i, val + 8, f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.suptitle('Modeling Targets — Class Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Demographics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

demo_cols = [
    ('q0026', 'Sexual orientation', '#1976D2'),
    ('q0028', 'Race/ethnicity', '#388E3C'),
    ('q0029', 'Education', '#F57C00'),
    ('q0024', 'Marital status', '#7B1FA2'),
    ('q0034', 'Household income', '#C62828'),
    ('q0009', 'Employment status', '#00838F'),
]

for ax, (col, title, color) in zip(axes.flatten(), demo_cols):
    counts = df[col].value_counts().head(7)
    ax.barh(counts.index[::-1], counts.values[::-1], color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Count', fontsize=9)

plt.suptitle('Respondent Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
ages = pd.to_numeric(df['q0027'], errors='coerce').dropna()
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ages, bins=30, color='#1976D2', edgecolor='white', alpha=0.85)
ax.axvline(ages.median(), color='#FF5722', linestyle='--', linewidth=1.5, label=f'Median: {ages.median():.0f}')
ax.set_xlabel('Age', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Age Distribution of Respondents', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Age — mean: {ages.mean():.1f}, median: {ages.median():.0f}, range: {ages.min():.0f}–{ages.max():.0f}")

## 5. Geographic Distribution

In [ ]:
region_map = {
    'Alabama':'South','Alaska':'West','Arizona':'West','Arkansas':'South',
    'California':'West','Colorado':'West','Connecticut':'Northeast','Delaware':'South',
    'Florida':'South','Georgia':'South','Hawaii':'West','Idaho':'West',
    'Illinois':'Midwest','Indiana':'Midwest','Iowa':'Midwest','Kansas':'Midwest',
    'Kentucky':'South','Louisiana':'South','Maine':'Northeast','Maryland':'South',
    'Massachusetts':'Northeast','Michigan':'Midwest','Minnesota':'Midwest',
    'Mississippi':'South','Missouri':'Midwest','Montana':'West','Nebraska':'Midwest',
    'Nevada':'West','New Hampshire':'Northeast','New Jersey':'Northeast',
    'New Mexico':'West','New York':'Northeast','North Carolina':'South',
    'North Dakota':'Midwest','Ohio':'Midwest','Oklahoma':'South','Oregon':'West',
    'Pennsylvania':'Northeast','Rhode Island':'Northeast','South Carolina':'South',
    'South Dakota':'Midwest','Tennessee':'South','Texas':'South','Utah':'West',
    'Vermont':'Northeast','Virginia':'South','Washington':'West',
    'West Virginia':'South','Wisconsin':'Midwest','Wyoming':'West',
    'District of Columbia':'South'
}

state_counts = df['q0030'].value_counts().head(15)
region_counts = df['q0030'].map(region_map).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(state_counts.index[::-1], state_counts.values[::-1], color='#1976D2', alpha=0.85)
axes[0].set_title('Top 15 States by Respondents', fontweight='bold')
axes[0].set_xlabel('Count')

axes[1].bar(region_counts.index, region_counts.values,
            color=['#1976D2', '#388E3C', '#F57C00', '#7B1FA2'], alpha=0.85)
axes[1].set_title('Respondents by Region', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(region_counts.values):
    axes[1].text(i, v + 3, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Lifestyle Frequency Patterns (Q7)

In [ ]:
# Correct Q7 labels per survey instrument
q7_labels = [
    'Ask friend (professional)', 'Ask friend (personal)', 'Physical affection — male friends',
    'Cry', 'Get in a physical fight', 'Sex with women', 'Sex with men', 'Watch sports',
    'Work out', 'See therapist', 'Feel lonely/isolated'
]
q7_cols = [c for c in df.columns if c.startswith('q0007')]
q7_order = ['Often', 'Sometimes', 'Rarely', 'Never, but open to it', 'Never, and not open to it']
q7_colors = ['#1565C0', '#42A5F5', '#FFD54F', '#EF9A9A', '#C62828']

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(q7_cols, q7_labels)):
    counts = df[col].value_counts().reindex(q7_order).fillna(0)
    pct = counts / counts.sum() * 100
    axes[i].bar(range(len(pct)), pct.values, color=q7_colors, edgecolor='white')
    axes[i].set_title(label, fontsize=9, fontweight='bold')
    axes[i].set_xticks(range(len(q7_order)))
    axes[i].set_xticklabels(['Often', 'Some-\ntimes', 'Rarely', 'Never\nopen', 'Never\nclosed'], fontsize=7)
    axes[i].set_ylabel('%', fontsize=8)
    axes[i].set_ylim(0, 80)

axes[-1].set_visible(False)
plt.suptitle('Q7: How Often Do You... (% of respondents)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Daily Worries (Q8)

In [ ]:
df_clean = pd.read_csv('../data/cleaned-responses.csv')

q8_labels = {
    'q0008_0001': 'Height', 'q0008_0002': 'Weight', 'q0008_0003': 'Hair/hairline',
    'q0008_0004': 'Physique', 'q0008_0005': 'Genitalia appearance',
    'q0008_0006': 'Clothing/style', 'q0008_0007': 'Sexual performance',
    'q0008_0008': 'Mental health', 'q0008_0009': 'Physical health',
    'q0008_0010': 'Finances/income', 'q0008_0011': 'Ability to provide',
    'q0008_0012': 'None of the above',
}

q8_cols = [c for c in df_clean.columns if c.startswith('q0008')]
q8_rates = df_clean[q8_cols].mean().sort_values(ascending=True)
q8_rates.index = [q8_labels[c] for c in q8_rates.index]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#C62828' if 'None' in label else '#1976D2' for label in q8_rates.index]
ax.barh(q8_rates.index, q8_rates.values, color=colors, alpha=0.85)
ax.set_xlabel('Proportion reporting as daily worry', fontsize=11)
ax.set_title('Q8: What Do Men Worry About Daily?', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

## 8. Source of Masculinity Ideas (Q4)

In [ ]:
q4_labels = {
    'q0004_0001': 'Father / father figure', 'q0004_0002': 'Mother / mother figure',
    'q0004_0003': 'Other family members', 'q0004_0004': 'Pop culture',
    'q0004_0005': 'Friends', 'q0004_0006': 'Other',
}

q4_cols = [c for c in df_clean.columns if c.startswith('q0004')]
q4_rates = df_clean[q4_cols].mean().sort_values(ascending=True)
q4_rates.index = [q4_labels[c] for c in q4_rates.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(q4_rates.index, q4_rates.values, color='#7B1FA2', alpha=0.85)
ax.set_xlabel('Proportion selecting (multi-select)', fontsize=11)
ax.set_title('Q4: Where Did You Get Your Ideas About Being a Good Man?', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

## 9. Relationship Behavior (Q18–Q20)

In [ ]:
q18_order = ['Always', 'Often', 'Sometimes', 'Rarely', 'Never']
q18_counts = df['q0018'].value_counts().reindex(q18_order).dropna()

q20_labels = {
    'q0020_0001': 'Read body language', 'q0020_0002': 'Ask verbally',
    'q0020_0003': 'Make a physical move', 'q0020_0004': 'Every situation is different',
    'q0020_0005': "It isn't always clear", 'q0020_0006': 'Other',
}
q20_cols = [c for c in df_clean.columns if c.startswith('q0020')]
q20_rates = df_clean[q20_cols].mean()
q20_rates.index = [q20_labels[c] for c in q20_rates.index]
q20_rates = q20_rates.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(q18_counts.index, q18_counts.values, color='#1976D2', alpha=0.85)
axes[0].set_title('Q18: How often tries to pay on dates', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(q18_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

axes[1].barh(q20_rates.index, q20_rates.values, color='#388E3C', alpha=0.85)
axes[1].set_title('Q20: How do you gauge interest? (multi-select)', fontweight='bold')
axes[1].set_xlabel('Proportion selecting')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

## 10. Correlation Heatmap — Q7 Lifestyle Frequencies

In [ ]:
q7_map = {'Often': 4, 'Sometimes': 3, 'Rarely': 2,
          'Never, but open to it': 1, 'Never, and not open to it': 0, 'No answer': np.nan}
q7_cols = [c for c in df.columns if c.startswith('q0007')]

q7_num = df[q7_cols].replace(q7_map)
# Correct column names per survey instrument
q7_num.columns = [
    'Ask friend (prof)', 'Ask friend (pers)', 'Physical affection',
    'Cry', 'Physical fight', 'Sex (women)', 'Sex (men)', 'Watch sports',
    'Work out', 'See therapist', 'Lonely/isolated'
]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(q7_num.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Q7 Lifestyle Frequency — Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()